In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider

import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset
df = pd.read_csv("superstore_dataset.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
# Doublons
print("Doublons:", df.duplicated().sum())
df = df.drop_duplicates()

# Valeurs manquantes
print(df.isnull().sum())

# Exemple traitement Postal Code
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

# Conversion dates
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df.info()

In [ ]:
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

df[['Sales', 'Profit', 'Profit Margin']].head()

In [ ]:
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12,6))
    
    if category == 'All':
        data = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(data.index.to_timestamp(), data.values, marker='o')
        plt.title("Monthly Sales - All Categories")
    else:
        data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(data['Date'], data['Sales'], marker='o')
        plt.title(f"Monthly Sales - {category}")

    plt.xticks(rotation=45)
    plt.grid()
    plt.show()

Dropdown(options=['All'] + list(df['Category'].unique()))
interact(plot_monthly_sales, category=['All'] + list(df['Category'].unique()));

In [ ]:
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=product_profit.values, y=product_profit.index, palette='viridis')

plt.title("Top 10 Most Profitable Products")
plt.xlabel("Profit")
plt.ylabel("Product")

for i, v in enumerate(product_profit.values):
    plt.text(v, i, f"{v:.0f}")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red')

plt.axhline(0, color='black', linestyle='--')
plt.title("Discount vs Profit")
plt.show()

In [ ]:
def dashboard():
    fig, axes = plt.subplots(2,2, figsize=(14,10))

    df.groupby('Category')['Sales'].sum().plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title("Sales by Category")

    df.groupby('Order Year')['Sales'].sum().plot(ax=axes[0,1])
    axes[0,1].set_title("Yearly Sales")

    state_sales.tail(10).plot(kind='barh', ax=axes[1,0])
    axes[1,0].set_title("Top States")

    sns.scatterplot(data=df, x='Discount', y='Profit', ax=axes[1,1])
    axes[1,1].set_title("Discount vs Profit")

    plt.tight_layout()
    plt.show()

dashboard()